# 🚁 UAV Acoustic Transformer - Multi-Drone Training Pipeline
This notebook automates data extraction, preprocessing, and GPU training for multiple drones.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the latest repository from GitHub
%cd /content
!git clone https://github.com/Adityaakumarr/UAV-Acoustic-Transformer.git
%cd UAV-Acoustic-Transformer
!pip install -r requirements.txt
!pip install rosbags soundfile

In [ ]:
# 3 & 4. Optimized Multi-Drone Data Transfer & Audio Extraction
# (Data is pulled directly from the 'paper-2 datasets' folder in your Google Drive)
DRIVE_PATH = "/content/drive/MyDrive/paper-2 datasets"
drones = ["Pham4", "Mavic3", "M300", "Avata"]

import os
!mkdir -p dataset/bags

# Step A: Unzip ground truths (These are small, so we do them together)
print("--- Step A: Extracting Ground Truths ---")
for drone in drones:
    print(f"Extracting {drone}.zip...")
    !unzip -q -o "{DRIVE_PATH}/{drone}.zip" -d dataset/

# Step B: Copy -> Extract -> Delete (ONE BY ONE TO SAVE SPACE!)
print("\n--- Step B: Extracting Audio (One by one) ---")
for drone in drones:
    print(f"\n🚀 Processing {drone}...")
    
    # 1. Copy only ONE bag file
    print(f"  [1/3] Copying {drone}.bag from Google Drive...")
    !rsync -ah --info=progress2 "{DRIVE_PATH}/{drone}.bag" dataset/bags/
    
    # 2. Extract Audio for that drone
    print(f"  [2/3] Extracting audio from {drone}.bag...")
    !python extract_audio.py --bag_dir dataset/bags --out_dir audio
    
    # 3. IMMEDIATELY delete the bag file to free up space!
    print(f"  [3/3] Deleting {drone}.bag to free up space! 🔥")
    !rm -f "dataset/bags/{drone}.bag"

print("\n✅ Audio Extraction Complete!")

In [ ]:
# 5. Feature Preprocessing (Generates 10-channel Acoustic .npy features)
!python feature_extraction.py


In [ ]:
# 6. Synchronize Ground Truth Labels
!python generate_aligned_labels.py --sequence Pham4
!python generate_aligned_labels.py --sequence Mavic3
!python generate_aligned_labels.py --sequence M300
!python generate_aligned_labels.py --sequence Avata


In [ ]:
# 7. Start Multi-Drone GPU Training
!python train.py

In [ ]:
# 8. Generate Evaluation Metrics and 3D Visualizations
!python post_train.py